# Algorithm Benchmark: CCG vs BDCP vs Hybrid CCG

Structured comparison of three solvers for the two-stage robust CFLP:
- **CCG** — Column-and-Constraint Generation (SOCP blocks, i-C&CG variant)
- **BDCP** — Benders Decomposition Cutting Plane (linear cuts only)
- **Hybrid CCG** — CCG + linear Benders cuts added alongside each SOCP block

**Setup** (run once from terminal before opening):
```bash
pip install -e .
```

## 0. Imports

In [ ]:
import time
import itertools
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

from rcflp.instance   import instancemaker
from rcflp.nominal    import solve_nominal
from rcflp.ccg        import solve_CCG
from rcflp.bdcp       import solve_BDCP
from rcflp.ccg_hybrid import solve_CCG_hybrid

pd.set_option('display.float_format', '{:.1f}'.format)
pd.set_option('display.max_columns', 30)
print('Imports OK')

## 1. Experiment configuration

Edit this cell to change the test grid. All three algorithms are run on every instance.

In [ ]:
# ── Instance grid ───────────────────────────────────────────────────────────
IJ_PAIRS   = [(10, 8), (15, 10)]   # (|I|, |J|) combinations
V_SCALES   = [0.75]                # value_max_scale
W_VALUES   = [100]                 # congestion_cost
GAMMAS     = [1, 2, 3]             # uncertainty budgets

Hn     = 2       # disruption levels
Rn     = 3       # capacity levels
BIG_M  = 10_000
OBJ_TOL = 0.01   # 1% optimality gap

# ── Per-algorithm time limits (seconds) ─────────────────────────────────────
def time_limit_for(In):
    if In <= 10:  return 300
    if In <= 15:  return 900
    return 1800

# ── CCG / Hybrid CCG master settings ────────────────────────────────────────
CCG_PARAMS = dict(
    master_mip_gap   = 0.015,
    master_time_limit= 100,
    n_scenarios      = 2,
    eps_e            = 0.009,
    alpha            = 0.8,
    beta             = 300,
)

# ── Build full parameter list ────────────────────────────────────────────────
PARAM_GRID = list(itertools.product(IJ_PAIRS, V_SCALES, W_VALUES, GAMMAS))
print(f'{len(PARAM_GRID)} configurations × 3 algorithms = {3*len(PARAM_GRID)} runs')

## 2. Run all algorithms

Results are collected in `rows` (one row per instance × algorithm) and also stored
in `runs` (one entry per instance, with full iter_logs for plotting).

In [ ]:
rows = []   # flat records for summary table
runs = []   # rich records for plotting

wall_start = time.time()

for idx, ((In, Jn), v, w, gamma) in enumerate(PARAM_GRID):
    tl = time_limit_for(In)
    tag = f'[{idx+1}/{len(PARAM_GRID)}] |I|={In} |J|={Jn} v={v} w={w} Γ={gamma}'
    print(f'\n{tag}')

    # ── Build instance and solve nominal ──────────────────────────────────
    inst  = instancemaker(In, Jn, Rn, v, w)
    nom   = solve_nominal(inst)
    x_nom = nom['x_jr']
    print(f'  nominal profit = {nom["profit"]:.0f}  |  time limit = {tl}s')

    common = dict(inst=inst, uncertainty_budget=gamma, Hn=Hn,
                  x_init=x_nom, tol=OBJ_TOL, big_M=BIG_M, time_limit=tl)

    algo_results = {}

    # ── CCG ───────────────────────────────────────────────────────────────
    print('  CCG ...', end=' ', flush=True)
    t0 = time.time()
    ccg = solve_CCG(**common, **CCG_PARAMS)
    print(f'{time.time()-t0:.1f}s | iters={ccg["n_iter"]} blocks={ccg["n_blocks"]} '
          f'conv={ccg["converged"]} profit={ccg["profit_LB"]:.0f}')
    algo_results['CCG'] = ccg

    # ── BDCP ──────────────────────────────────────────────────────────────
    print('  BDCP ...', end=' ', flush=True)
    t0 = time.time()
    bdcp = solve_BDCP(**common)
    print(f'{time.time()-t0:.1f}s | iters={bdcp["n_iter"]} '
          f'conv={bdcp["converged"]} profit={bdcp["profit_LB"]:.0f}')
    algo_results['BDCP'] = bdcp

    # ── Hybrid CCG ────────────────────────────────────────────────────────
    print('  Hybrid CCG ...', end=' ', flush=True)
    t0 = time.time()
    hyb = solve_CCG_hybrid(**common, **CCG_PARAMS)
    print(f'{time.time()-t0:.1f}s | iters={hyb["n_iter"]} blocks={hyb["n_blocks"]} '
          f'benders_cuts={hyb["n_benders"]} conv={hyb["converged"]} profit={hyb["profit_LB"]:.0f}')
    algo_results['Hybrid'] = hyb

    # ── Check profit agreement ────────────────────────────────────────────
    profits = {k: round(v['profit_LB'], 0) for k, v in algo_results.items()}
    max_diff = max(profits.values()) - min(profits.values())
    if max_diff > 10:
        print(f'  ⚠  PROFIT MISMATCH: {profits}  diff={max_diff:.0f}')

    # ── Collect flat rows ─────────────────────────────────────────────────
    base = dict(I=In, J=Jn, v=v, w=w, gamma=gamma, nom_profit=nom['profit'])
    for algo, res in algo_results.items():
        row = {**base, 'algo': algo,
               'profit':    res['profit_LB'],
               'runtime':   res['runtime'],
               'n_iter':    res['n_iter'],
               'n_blocks':  res.get('n_blocks', None),
               'n_benders': res.get('n_benders', None),
               'converged': res['converged'],
               'gap_final': res['iter_log'][-1]['gap_pct'] if res['iter_log'] else None,
               't_master_total': sum(d['t_master'] for d in res['iter_log']),
               't_sub_total':    sum(d['t_sub']    for d in res['iter_log']),
        }
        rows.append(row)

    runs.append(dict(base=base, results=algo_results))

print(f'\nDone. Total wall time: {(time.time()-wall_start)/60:.1f} min')

## 3. Summary table

In [ ]:
df = pd.DataFrame(rows)

# Pivot to wide format: one row per instance, columns per algorithm
pivot_cols = ['profit', 'runtime', 'n_iter', 'n_blocks', 'converged', 'gap_final',
              't_master_total', 't_sub_total']
wide = df.pivot_table(index=['I','J','v','w','gamma'],
                      columns='algo', values=pivot_cols, aggfunc='first')
wide.columns = ['_'.join(c).strip() for c in wide.columns]
wide = wide.reset_index()

# Flag profit mismatches
for col in ['profit_CCG', 'profit_BDCP', 'profit_Hybrid']:
    if col not in wide.columns:
        wide[col] = np.nan
wide['mismatch'] = (wide[['profit_CCG','profit_BDCP','profit_Hybrid']].max(axis=1)
                  - wide[['profit_CCG','profit_BDCP','profit_Hybrid']].min(axis=1)) > 10

def highlight_mismatch(row):
    return ['background-color: #ffdddd' if row.get('mismatch') else ''] * len(row)

display(wide.style.apply(highlight_mismatch, axis=1).format(precision=1))

## 4. Head-to-head: runtime and iterations

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(14, 9))
pairs = [('CCG', 'BDCP'), ('CCG', 'Hybrid'), ('BDCP', 'Hybrid')]
colors = df['gamma'].map({g: c for g, c in zip(sorted(df['gamma'].unique()),
                          ['#4C72B0','#DD8452','#55A868','#C44E52'])})

for col, (a, b) in enumerate(pairs):
    for row, metric in enumerate(['runtime', 'n_iter']):
        ax = axes[row, col]
        xa = df[df['algo'] == a][['I','J','v','w','gamma', metric]].rename(columns={metric: 'x'})
        xb = df[df['algo'] == b][['I','J','v','w','gamma', metric]].rename(columns={metric: 'y'})
        merged = xa.merge(xb, on=['I','J','v','w','gamma'])
        sc = ax.scatter(merged['x'], merged['y'],
                        c=merged['gamma'], cmap='viridis', s=80, alpha=0.8, zorder=3)
        lim = max(merged['x'].max(), merged['y'].max()) * 1.1
        ax.plot([0, lim], [0, lim], 'k--', lw=1, alpha=0.5)
        ax.set_xlabel(f'{a}  {metric}', fontsize=10)
        ax.set_ylabel(f'{b}  {metric}', fontsize=10)
        label = 'Runtime (s)' if metric == 'runtime' else 'Iterations'
        ax.set_title(f'{a} vs {b} — {label}', fontsize=11)
        ax.grid(ls='--', alpha=0.4)
        plt.colorbar(sc, ax=ax, label='Γ')

fig.tight_layout()
fig.savefig('fig_headtohead.pdf', bbox_inches='tight')
plt.show()

## 5. Master vs subproblem time breakdown

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4), sharey=True)
algos = ['CCG', 'BDCP', 'Hybrid']

for ax, algo in zip(axes, algos):
    sub = df[df['algo'] == algo].copy()
    total = sub['t_master_total'] + sub['t_sub_total']
    pct_master = 100 * sub['t_master_total'] / total
    pct_sub    = 100 * sub['t_sub_total']    / total
    x = range(len(sub))
    ax.bar(x, pct_master, label='Master', color='#4C72B0', alpha=0.8)
    ax.bar(x, pct_sub,    bottom=pct_master, label='Subproblem',
           color='#DD8452', alpha=0.8)
    labels = [f"I{r.I}J{r.J}\nΓ{r.gamma}" for r in sub.itertuples()]
    ax.set_xticks(list(x))
    ax.set_xticklabels(labels, fontsize=7)
    ax.set_title(algo, fontsize=12)
    ax.set_ylabel('% of total time' if algo == 'CCG' else '')
    ax.set_ylim(0, 105)
    ax.legend(fontsize=9)
    ax.grid(axis='y', ls='--', alpha=0.4)

fig.tight_layout()
fig.savefig('fig_time_breakdown.pdf', bbox_inches='tight')
plt.show()

## 6. Convergence plots (UB and LB over time)

One figure per instance. Change `RUN_IDX` to select a specific instance.

In [ ]:
ALGO_STYLES = {
    'CCG':    dict(color='#4C72B0', ls='-',  lw=1.8),
    'BDCP':   dict(color='#DD8452', ls='--', lw=1.8),
    'Hybrid': dict(color='#55A868', ls='-.',  lw=1.8),
}

for run in runs:
    b = run['base']
    fig, axes = plt.subplots(1, 2, figsize=(13, 4))
    fig.suptitle(f"|I|={b['I']} |J|={b['J']}  v={b['v']}  w={b['w']}  Γ={b['gamma']}",
                 fontsize=12)

    for algo, res in run['results'].items():
        log = res['iter_log']
        if not log:
            continue
        elapsed = [d['elapsed'] for d in log]
        ub  = [-d['UB'] for d in log]
        lb  = [-d['LB'] for d in log]
        gap = [d['gap_pct'] for d in log]
        sty = ALGO_STYLES[algo]

        axes[0].plot(elapsed, ub, label=f'{algo} UB', **sty)
        axes[0].plot(elapsed, lb, label=f'{algo} LB', alpha=0.5, **sty)
        axes[1].plot(elapsed, gap, label=algo, **sty)

    axes[0].set_xlabel('Time (s)')
    axes[0].set_ylabel('Profit')
    axes[0].set_title('Upper and lower bounds')
    axes[0].legend(fontsize=8, ncol=2)
    axes[0].grid(ls='--', alpha=0.4)

    axes[1].axhline(1.0, color='k', lw=0.8, ls=':', label='1% target')
    axes[1].set_xlabel('Time (s)')
    axes[1].set_ylabel('Gap (%)')
    axes[1].set_title('Optimality gap over time')
    axes[1].legend(fontsize=9)
    axes[1].grid(ls='--', alpha=0.4)

    fig.tight_layout()
    fname = f"fig_conv_I{b['I']}J{b['J']}_v{b['v']}_w{b['w']}_G{b['gamma']}.pdf"
    fig.savefig(fname, bbox_inches='tight')
    plt.show()

## 7. Per-iteration master time (shows whether master cost grows with blocks)

In [ ]:
for run in runs:
    b = run['base']
    fig, ax = plt.subplots(figsize=(10, 4))
    fig.suptitle(f"|I|={b['I']} |J|={b['J']}  v={b['v']}  w={b['w']}  Γ={b['gamma']}",
                 fontsize=12)

    for algo, res in run['results'].items():
        log = res['iter_log']
        if not log:
            continue
        iters      = [d['iter']     for d in log]
        t_master   = [d['t_master'] for d in log]
        n_blocks   = [d.get('n_blocks', 0) for d in log]
        sty = ALGO_STYLES[algo]
        ax.plot(iters, t_master, label=algo, marker='o', markersize=3, **sty)

    ax.set_xlabel('Iteration')
    ax.set_ylabel('Master solve time (s)')
    ax.set_title('Per-iteration master cost')
    ax.legend(fontsize=9)
    ax.grid(ls='--', alpha=0.4)
    fig.tight_layout()
    plt.show()

## 8. Aggregate summary statistics

In [ ]:
summary = (df.groupby('algo')
             .agg(
                 instances   = ('runtime',   'count'),
                 converged   = ('converged', 'sum'),
                 runtime_med = ('runtime',   'median'),
                 runtime_max = ('runtime',   'max'),
                 iters_med   = ('n_iter',    'median'),
                 iters_max   = ('n_iter',    'max'),
                 pct_master  = ('t_master_total',
                                lambda x: 100 * x.sum() /
                                (x.sum() + df.loc[x.index, 't_sub_total'].sum())),
             )
             .rename(columns={
                 'runtime_med': 'Runtime median (s)',
                 'runtime_max': 'Runtime max (s)',
                 'iters_med':   'Iters median',
                 'iters_max':   'Iters max',
                 'pct_master':  '% time in master',
             })
)
display(summary.style.format(precision=1))

## 9. Save raw results to CSV

In [ ]:
df.to_csv('benchmark_results.csv', index=False)
print('Saved benchmark_results.csv')
df